<a href="https://colab.research.google.com/github/Takumi173/Test/blob/main/Dataset_JSON_Reviewer_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 準備

## 処理用にデータを結合

In [12]:
# データのコピー
!git clone https://github.com/cdisc-org/sdtm-adam-pilot-project.git

Cloning into 'sdtm-adam-pilot-project'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 224 (delta 64), reused 220 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 24.51 MiB | 7.42 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (87/87), done.


In [13]:
# 使用するjsonデータとdefine.xmlを新規フォルダにコピーする

import os
import shutil
import json

source_dir = "sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm"
json_dir   = "json_files"
define_dir = "define_xml"

if not os.path.exists(json_dir):
    os.makedirs(json_dir)

if not os.path.exists(define_dir):
    os.makedirs(define_dir)

for root, _, files in os.walk(source_dir):
  for file in files:
    if file.endswith(".json"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(json_dir, file)
      shutil.copy(source_path, target_path)
    if file.endswith("define.xml"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(define_dir, file)
      shutil.copy(source_path, target_path)

In [14]:
# jsonファイルをリスト形式に結合したファイル（dataset_list.json）を作成

dataset_list = []
for filename in os.listdir(json_dir):
  if filename.endswith(".json"):
    with open(os.path.join(json_dir, filename), "r") as f:
      try:
        json_data = json.load(f)
        dataset_list.append(json_data)
      except json.JSONDecodeError as e:
        print(f"Error decoding JSON in file {filename}: {e}")

with open("dataset_list.json", "w") as f:
  json.dump(dataset_list, f)


## 症例フィルタリング関数の定義

In [15]:
def filter_data(data, target_usubjids):
    """
    複数のドメインデータを含むリストから、指定されたUSUBJIDのrowsのみを抽出して新しいJSONファイルに保存する。
    入力データがリストでない場合はエラーメッセージを出力する。
    データ構造は、"columns" 内の "name" が "USUBJID" の列を持つことを前提とする。

    Args:
        data (list): ドメインを結合させたのリスト。リストでない場合はエラーとなる。
        output_file (str): 出力するJSONファイル名。
        target_usubjids (list): 残したいUSUBJIDのリスト。
    """
    if not isinstance(data, list):
        print("エラー：入力データはJSONオブジェクトのリストである必要があります。")
        return

    filtered_data_list = []
    for item in data:
        usubjid_index = -1
        if 'columns' in item:
            for i, col in enumerate(item['columns']):
                if 'name' in col and col['name'] == 'USUBJID':
                    usubjid_index = i
                    break

        if usubjid_index == -1:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'name' が 'USUBJID' の列が見つかりません。スキップします。")
            filtered_data_list.append(item)
            continue

        if 'rows' in item:
            filtered_rows = [
                row for row in item['rows'] if len(row) > usubjid_index and row[usubjid_index] in target_usubjids
            ]
            new_data = item.copy()
            new_data['rows'] = filtered_rows
            new_data['records'] = len(filtered_rows)
            filtered_data_list.append(new_data)
        else:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'rows' が見つかりません。スキップします。")
            filtered_data_list.append(item)

    return filtered_data_list



# 実行テスト
# with open('dataset_list.json', 'r') as f:
#   data = json.load(f)
#
# target_ids = ['01-701-1211']
# output_filename = 'filtered_list.json'
#
# filtered_data_list = filter_data(data, target_ids)
#
# with open(output_filename, 'w') as f:
#   json.dump(filtered_data_list, f)
#
# print(f"処理完了：'{output_filename}' に USUBJID が {target_ids} のデータを出力しました。")

## データ書き換え関数の定義

In [16]:
def data_update(data, target_domain, target_usubjid, target_seq, target_variable, new_value):
    """
    指定されたUSUBJIDを持つレコードの指定された変数を書き換えます。
    {target_domain}SEQが存在する場合はそれもキーとして使用します。
    元のデータは変更せず、新しいデータ構造を返します。

    Args:
        data (list): データ全体のリスト。指定されたtarget_domainのデータセットを含むことを想定します。
        target_domain (str): 対象のドメイン名（例: "CM"）。
        target_usubjid (str): 書き換えたいレコードのUSUBJID。
        target_seq (int): 書き換えたいレコードの{target_domain}SEQの値（存在しない場合は無視されます）。
        target_variable (str): 書き換えたい変数の名前（例: "CMTRT"）。
        new_value (any): 新しい変数の値。

    Returns:
        list: 指定された変数が更新された新しいデータ全体のリスト。
              該当するレコードが見つからなかった場合、元のデータのコピーを返します。
    """
    updated_data = []
    seqname = target_domain + 'SEQ'

    for dataset in data:
        updated_dataset = dataset.copy()
        if updated_dataset.get("itemGroupOID") == target_domain:
            updated_rows = []
            found = False
            usubjid_index = -1
            seq_index = -1
            variable_index = -1
            has_seq = False

            for i, col in enumerate(updated_dataset["columns"]):
                if col["name"] == "USUBJID":
                    usubjid_index = i
                elif col["name"] == seqname:
                    seq_index = i
                    has_seq = True
                elif col["name"] == target_variable:
                    variable_index = i

            if usubjid_index != -1 and variable_index != -1:
                for row in dataset["rows"]:
                    updated_row = list(row)  # 行をコピーして変更
                    usubjid_match = updated_row[usubjid_index] == target_usubjid
                    seq_match = True
                    if has_seq and seq_index != -1:
                        seq_match = (len(updated_row) > seq_index and updated_row[seq_index] == target_seq)
                    elif has_seq:
                        print(f"警告: '{target_domain}' データセットに '{seqname}' 列が見つかりましたが、インデックスが無効です。USUBJIDのみをキーとして使用します。")

                    if usubjid_match and seq_match:
                        updated_row[variable_index] = new_value
                        if has_seq and seq_index != -1:
                            print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' の '{target_variable}' を '{new_value}' に更新しました。")
                        else:
                            print(f"USUBJID '{target_usubjid}' の '{target_variable}' を '{new_value}' に更新しました。")
                        found = True
                    updated_rows.append(updated_row)
                updated_dataset["rows"] = updated_rows
            elif updated_dataset.get("itemGroupOID") == target_domain:
                print(f"'{target_domain}' データセットに 'USUBJID' または '{target_variable}' 列が見つかりませんでした。")

            updated_data.append(updated_dataset)
            if not found and updated_dataset.get("itemGroupOID") == target_domain:
                if has_seq and seq_index != -1:
                    print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' に該当するレコードが見つかりませんでした。")
                else:
                    print(f"USUBJID '{target_usubjid}' に該当するレコードが見つかりませんでした。")
        else:
            updated_data.append(updated_dataset)

    if not any(d.get("itemGroupOID") == target_domain for d in data):
        print(f"{target_domain} データセットが見つかりませんでした。")

    return updated_data

# 書き換えテスト
# updated_data = data_update(filtered_data_list, "DM", "01-701-1211", 0, "AGE", 49)
# updated_data = data_update(updated_data, "CM", "01-701-1211", 3, "CMTRT", "New Drug 123456789")
# updated_data = data_update(updated_data, "CM", "01-701-1211", 0, "CMDOSE", 123)

## データ比較関数の定義

In [17]:
from typing import List, Dict, Any

def compare_data(old_data: List[Dict[str, Any]], new_data: List[Dict[str, Any]]) -> None:
    """
    2つのデータリストの更新差分を人間が読みやすい形式で出力します。

    Args:
        old_data: 旧データリスト。
        new_data: 新データリスト。
    """

    def create_row_dict(item_group: Dict[str, Any], row: List[Any]) -> Dict[str, Any]:
        """rowデータをキー付きの辞書に変換する"""
        row_dict = {}
        for i, column in enumerate(item_group['columns']):
            row_dict[column['name']] = row[i]
        return row_dict

    def get_key_values(item_group_oid: str, row_dict: Dict[str, Any]) -> Dict[str, Any]:
        """データのキーとなる値を抽出する"""
        key_values = {'USUBJID': row_dict.get('USUBJID')}
        seq_key = f"{item_group_oid}SEQ"
        if seq_key in row_dict:
            key_values[seq_key] = row_dict[seq_key]
        return key_values

    def format_key(key_values: Dict[str, Any]) -> str:
        """キー値を人間が読みやすい文字列に整形する"""
        parts = []
        for key, value in key_values.items():
            if value is not None:
                parts.append(f"{key} = {value}")
        return ", ".join(parts)

    old_data_by_group = {item['itemGroupOID']: item for item in old_data}
    new_data_by_group = {item['itemGroupOID']: item for item in new_data}

    all_group_oids = set(old_data_by_group.keys()) | set(new_data_by_group.keys())

    for group_oid in sorted(list(all_group_oids)):
        print(f"--- ItemGroupOID: {group_oid} ---")
        old_group = old_data_by_group.get(group_oid)
        new_group = new_data_by_group.get(group_oid)

        old_rows_by_key = {}
        if old_group and 'rows' in old_group:
            for row in old_group['rows']:
                row_dict = create_row_dict(old_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    old_rows_by_key[format_key(key_values)] = row_dict

        new_rows_by_key = {}
        if new_group and 'rows' in new_group:
            for row in new_group['rows']:
                row_dict = create_row_dict(new_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    new_rows_by_key[format_key(key_values)] = row_dict

        old_keys = set(old_rows_by_key.keys())
        new_keys = set(new_rows_by_key.keys())

        # 追加されたデータ
        added_keys = new_keys - old_keys
        for key in sorted(list(added_keys)):
            print(f"{key}:")
            print("  Added")
            for item_key, old_value in sorted(new_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 削除されたデータ
        removed_keys = old_keys - new_keys
        for key in sorted(list(removed_keys)):
            print(f"{key}:")
            print("  Deleted")
            for item_key, old_value in sorted(old_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 更新されたデータ
        common_keys = old_keys & new_keys
        for key in sorted(list(common_keys)):
            if old_rows_by_key[key] != new_rows_by_key[key]:
                print(f"{key}:")
                print("  Updated:")
                old_row = old_rows_by_key[key]
                new_row = new_rows_by_key[key]
                for item_key in sorted(list(set(old_row.keys()) | set(new_row.keys()))):
                    old_value = old_row.get(item_key)
                    new_value = new_row.get(item_key)
                    if old_value != new_value:
                        print(f"    {item_key}: {old_value!r} -> {new_value!r}")
                print()


# 比較テスト
# compare_data(filtered_data_list, updated_data)

In [18]:
import json

usubjids = set()
for dataset in dataset_list:
    if 'columns' in dataset:
        for i, col in enumerate(dataset['columns']):
            if 'name' in col and col['name'] == 'USUBJID':
                if 'rows' in dataset:
                    for row in dataset['rows']:
                        if len(row) > i:
                            usubjids.add(row[i])

print(list(usubjids))


['01-703-1403', '01-701-1033', '01-713-1179', '01-716-1441', '01-717-1174', '01-701-1240', '01-701-1369', '01-710-1137', '01-710-1358', '01-715-1405', '01-704-1435', '01-718-1250', '01-704-1325', '01-708-1378', '01-703-1439', '01-709-1020', '01-711-1284', '01-703-1042', '01-716-1094', '01-707-1430', '01-704-1008', '01-704-1010', '01-710-1264', '01-703-1295', '01-703-1197', '01-704-1351', '01-708-1336', '01-701-1111', '01-705-1243', '01-701-1034', '01-701-1181', '01-706-1041', '01-708-1342', '01-701-1234', '01-708-1347', '01-709-1301', '01-710-1380', '01-718-1328', '01-710-1376', '01-716-1103', '01-717-1004', '01-708-1348', '01-703-1076', '01-710-1053', '01-701-1203', '01-710-1300', '01-704-1114', '01-711-1022', '01-704-1260', '01-706-1384', '01-710-1408', '01-716-1071', '01-701-1442', '01-704-1120', '01-716-1063', '01-716-1311', '01-701-1015', '01-708-1216', '01-703-1299', '01-708-1286', '01-701-1153', '01-709-1081', '01-708-1067', '01-703-1086', '01-705-1112', '01-716-1298', '01-701-1

# データの書き換え

In [19]:
Target_data = [
["DM", "01-703-1096",   0, "AGE", 49],
["LB", "01-703-1042",   3, "LBORRES", "135"],
["LB", "01-703-1042",   4, "LBORRES", "145"],
["LB", "01-703-1086",  37, "LBORRES", "1"],
["LB", "01-703-1086",  72, "LBORRES", "1.2"],
["LB", "01-703-1086", 102, "LBORRES", "1.1"],
["LB", "01-703-1086", 132, "LBORRES", "1"],
["LB", "01-703-1086", 162, "LBORRES", "1.3"],
["LB", "01-703-1086", 197, "LBORRES", "0.9"],
["LB", "01-703-1086", 232, "LBORRES", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESC", "135"],
["LB", "01-703-1042",   4, "LBSTRESC", "145"],
["LB", "01-703-1086",  37, "LBSTRESC", "1"],
["LB", "01-703-1086",  72, "LBSTRESC", "1.2"],
["LB", "01-703-1086", 102, "LBSTRESC", "1.1"],
["LB", "01-703-1086", 132, "LBSTRESC", "1"],
["LB", "01-703-1086", 162, "LBSTRESC", "1.3"],
["LB", "01-703-1086", 197, "LBSTRESC", "0.9"],
["LB", "01-703-1086", 232, "LBSTRESC", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESN", 135],
["LB", "01-703-1042",   4, "LBSTRESN", 145],
["LB", "01-703-1086",  37, "LBSTRESN", 1],
["LB", "01-703-1086",  72, "LBSTRESN", 1.2],
["LB", "01-703-1086", 102, "LBSTRESN", 1.1],
["LB", "01-703-1086", 132, "LBSTRESN", 1],
["LB", "01-703-1086", 162, "LBSTRESN", 1.3],
["LB", "01-703-1086", 197, "LBSTRESN", 0.9],
["LB", "01-703-1086", 232, "LBSTRESN", 0.8],
["LB", "01-703-1042",   3, "LBNRIND", "HIGH"],
["LB", "01-703-1042",   4, "LBNRIND", "HIGH"],
["LB", "01-703-1086",  37, "LBNRIND", "LOW"],
["LB", "01-703-1086",  72, "LBNRIND", "LOW"],
["LB", "01-703-1086", 102, "LBNRIND", "LOW"],
["LB", "01-703-1086", 132, "LBNRIND", "LOW"],
["LB", "01-703-1086", 162, "LBNRIND", "LOW"],
["LB", "01-703-1086", 197, "LBNRIND", "LOW"],
["LB", "01-703-1086", 232, "LBNRIND", "LOW"],
["MH", "01-701-1097",   1, "MHTERM", "Loss of consciousness (Passed out)"],
["MH", "01-701-1097",   1, "MHSTDTC", "2023-01-01"],
["MH", "01-701-1111",   1, "MHTERM", "HEARING LOSS"],
["MH", "01-701-1180",   1, "MHTERM", "DEPRESSION (ANXIETY)"],
["MH", "01-702-1082",   1, "MHTERM", "Premenstrual pain"],
["MH", "01-703-1076",   1, "MHTERM", "Atrioventricular block (scheduled cardiac pacemaker insertion)"],
["MH", "01-703-1279",   1, "MHTERM", "schizophreniform disorders"],
["MH", "01-703-1299",   1, "MHTERM", "Cyclothymic disorder"],
["VS", "01-701-1047",  17, "VSORRES", "121"],
["VS", "01-701-1047",  18, "VSORRES", "124"],
["VS", "01-701-1047",  66, "VSORRES", "185"],
["VS", "01-701-1047",  67, "VSORRES", "183"],
["VS", "01-701-1383",  37, "VSORRES", "98"],
["VS", "01-701-1383", 122, "VSORRES", "160"],
["VS", "01-701-1387",   1, "VSORRES", "146"],
["VS", "01-701-1387",  32, "VSORRES", "72"],
["VS", "01-701-1047",  17, "VSSTRESC", "121"],
["VS", "01-701-1047",  18, "VSSTRESC", "124"],
["VS", "01-701-1047",  66, "VSSTRESC", "185"],
["VS", "01-701-1047",  67, "VSSTRESC", "183"],
["VS", "01-701-1383",  37, "VSSTRESC", "98"],
["VS", "01-701-1383", 122, "VSSTRESC", "160"],
["VS", "01-701-1387",   1, "VSSTRESC", "146"],
["VS", "01-701-1387",  32, "VSSTRESC", "72"],
["VS", "01-701-1047",  17, "VSSTRESN", 121],
["VS", "01-701-1047",  18, "VSSTRESN", 124],
["VS", "01-701-1047",  66, "VSSTRESN", 185],
["VS", "01-701-1047",  67, "VSSTRESN", 183],
["VS", "01-701-1383",  37, "VSSTRESN", 98],
["VS", "01-701-1383", 122, "VSSTRESN", 160],
["VS", "01-701-1387",   1, "VSSTRESN", 146],
["VS", "01-701-1387",  32, "VSSTRESN", 72],
["EX", "01-701-1148",   2, "EXDOSE", 82],
["EX", "01-701-1148",   3, "EXDOSE", 216],
["EX", "01-703-1258",   2, "EXDOSE", 27],
["CM", "01-701-1146",  29, "CMTRT", "PAROXETINE"],
["QS", "01-701-1023",1010, "QSORRES", "PRESENT"],
["QS", "01-701-1023",1012, "QSORRES", "PRESENT"],
["QS", "01-701-1111",5004, "QSORRES", "4"],
["QS", "01-701-1111",5019, "QSORRES", "4"],
["QS", "01-701-1111",5012, "QSORRES", "4"],
["QS", "01-701-1111",5027, "QSORRES", "4"],
["QS", "01-701-1118",6002, "QSORRES", "MARKED IMPROVEMENT"],
["QS", "01-701-1118",6003, "QSORRES", "MARKED WORSENING"],
["QS", "01-701-1181",4018, "QSORRES", "Y"],
["QS", "01-701-1181",4058, "QSORRES", "Y"],
["QS", "01-701-1181",4019, "QSORRES", "Y"],
["QS", "01-701-1181",4059, "QSORRES", "Y"],
["QS", "01-701-1181",4020, "QSORRES", "Y"],
["QS", "01-701-1023",1010, "QSSTRESC", "2"],
["QS", "01-701-1023",1012, "QSSTRESC", "2"],
["QS", "01-701-1111",5004, "QSSTRESC", "4"],
["QS", "01-701-1111",5019, "QSSTRESC", "4"],
["QS", "01-701-1111",5012, "QSSTRESC", "4"],
["QS", "01-701-1111",5027, "QSSTRESC", "4"],
["QS", "01-701-1118",6002, "QSSTRESC", "1"],
["QS", "01-701-1118",6003, "QSSTRESC", "7"],
["QS", "01-701-1181",4018, "QSSTRESC", "1"],
["QS", "01-701-1181",4058, "QSSTRESC", "1"],
["QS", "01-701-1181",4019, "QSSTRESC", "1"],
["QS", "01-701-1181",4059, "QSSTRESC", "1"],
["QS", "01-701-1181",4020, "QSSTRESC", "1"],
["QS", "01-701-1023",1010, "QSSTRESN", 2],
["QS", "01-701-1023",1012, "QSSTRESN", 2],
["QS", "01-701-1111",5004, "QSSTRESN", 4],
["QS", "01-701-1111",5019, "QSSTRESN", 4],
["QS", "01-701-1111",5012, "QSSTRESN", 4],
["QS", "01-701-1111",5027, "QSSTRESN", 4],
["QS", "01-701-1118",6002, "QSSTRESN", 1],
["QS", "01-701-1118",6003, "QSSTRESN", 7],
["QS", "01-701-1181",4018, "QSSTRESN", 1],
["QS", "01-701-1181",4058, "QSSTRESN", 1],
["QS", "01-701-1181",4019, "QSSTRESN", 1],
["QS", "01-701-1181",4059, "QSSTRESN", 1],
["QS", "01-701-1181",4020, "QSSTRESN", 1],
["QS", "01-701-1118",6001, "QSDTC", "2014-07-08"],
["QS", "01-701-1118",6001, "QSDY", 119],
["AE", "01-701-1015",   3, "AESER", "Y"],
["AE", "01-701-1015",   3, "AESHOSP", "Y"],
["AE", "01-701-1015",   3, "AESTDTC", "2014-01-11"],
["AE", "01-701-1015",   3, "AEENDTC", "2014-01-09"],
["AE", "01-701-1015",   3, "AESTDY", 10],
["AE", "01-701-1015",   3, "AEENDY", 8],
["AE", "01-701-1028",   1, "AETERM", "PARKINSON'S DISEASE"],
["AE", "01-701-1028",   1, "AESTDTC", "2013-07-01"],
["AE", "01-701-1028",   1, "AESTDY", -17],
["AE", "01-701-1034",   2, "AETERM", "MALIGNANT HYPERTENSION"],
["AE", "01-701-1047",   4, "AETERM", "HYPERTENSION"],
["AE", "01-701-1363",   1, "AESTDTC", "2013-06-15"],
["AE", "01-701-1363",   1, "AEENDTC", "2013-06-14"],
["AE", "01-701-1363",   1, "AESTDY", 17],
["AE", "01-701-1363",   1, "AEENDY", 16],
["AE", "01-701-1047",   3, "AEENDTC", "2013-03-05"],
["AE", "01-701-1047",   3, "AEENDY", 22],
["AE", "01-701-1383",  12, "AETERM", "BLOOD PRESSURE INCREASED"],
["AE", "01-701-1153",   2, "AEACN", "DRUG WITHDRAWN"],
["AE", "01-701-1180",   6, "AETERM", "SUDDEN DEATH"],
["AE", "01-703-1258",   2, "AESEV", "SEVERE"],
["AE", "01-703-1258",   2, "AESTDTC", "2012-08-01"],
["AE", "01-703-1258",   2, "AEENDTC", "2012-10-01"],
["AE", "01-703-1258",   2, "AESTDY", 13],
["AE", "01-703-1258",   2, "AEENDY", 74],
["AE", "01-703-1258",   5, "AESEV", "MODERATE"],
["AE", "01-703-1258",   5, "AESER", "Y"],
["AE", "01-703-1258",   5, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-703-1258",   5, "AESLIFE", "Y"],
["AE", "01-703-1258",   5, "AESTDTC", "2012-10-02"],
["AE", "01-703-1258",   5, "AEENDTC", "2012-12-31"],
["AE", "01-703-1258",   2, "AESTDY", 75],
["AE", "01-703-1258",   2, "AEENDY", 165],
["AE", "01-703-1335",   1, "AETERM", "MULTIPLE SCLEROSIS RELAPSE"],
["AE", "01-703-1335",   1, "AESTDTC", "2014-04-01"],
["AE", "01-703-1335",   1, "AEENDTC", "2014-05-01"],
["AE", "01-703-1335",   1, "AESTDY", 15],
["AE", "01-703-1335",   1, "AEENDY", 46],
["AE", "01-703-1403",   2, "AETERM", "MYASTHENIA GRAVIS AGGRAVATED"],
["AE", "01-704-1008",   1, "AETERM", "TREMOR IN HANDS, LEGS"],
["AE", "01-704-1008",   1, "AEREL", "NONE"],
["AE", "01-704-1008",   1, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   1, "AESTDY", -225],
["AE", "01-704-1008",   3, "AETERM", "MUSCLE STIFFNESS"],
["AE", "01-704-1008",   3, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   3, "AESTDY", -225],
["AE", "01-704-1008",   2, "AETERM", "SLOWNESS of MOVEMENT"],
["AE", "01-704-1008",   2, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   2, "AESTDY", -225],
["AE", "01-704-1009",   6, "AETERM", "CHRONIC KIDNEY DISEASE"],
["AE", "01-704-1009",   6, "AESER", "Y"],
["AE", "01-704-1009",   6, "AESLIFE", "Y"],
["AE", "01-704-1010",   1, "AETERM", "DIABETES MELLITUS"],
["AE", "01-704-1010",   1, "AESER", "Y"],
["AE", "01-704-1010",   1, "AESLIFE", "Y"],
["AE", "01-704-1017",   4, "AETERM", "LATE EFFECTS OF CEREBRAL INFRACTION"],
["AE", "01-704-1017",   4, "AESEV", "SEVERE",],
["AE", "01-704-1017",   4, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   4, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   4, "AESTDY", 14],
["AE", "01-704-1017",   4, "AEENDY", 44],
["AE", "01-704-1017",   3, "AETERM", "BRAIN DEATH"],
["AE", "01-704-1017",   3, "AESEV", "SEVERE",],
["AE", "01-704-1017",   3, "AESTDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AESTDY", 44],
["AE", "01-704-1017",   3, "AEENDY", 44],
["AE", "01-704-1017",   1, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-704-1017",   1, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   1, "AEENDTC", "2013-11-19"],
["AE", "01-704-1017",   1, "AESTDY", 14],
["AE", "01-704-1017",   1, "AEENDY", 45],
["AE", "01-704-1017",   1, "AEACN", "DRUG WITHDRAWN"]
]

dataset_list_updated = dataset_list

for l in Target_data:
  #print(l)
  dataset_list_updated = data_update(dataset_list_updated, l[0], l[1], l[2], l[3], l[4])

USUBJID '01-703-1096' の 'AGE' を '49' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBORRES' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBORRES' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBORRES' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBORRES' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '162' の 'LBORRES' を '1.3' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '197' の 'LBORRES' を '0.9' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '232' の 'LBORRES' を '0.8' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBSTRESC' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBSTRESC' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBSTRESC' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBSTRESC' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBSTRESC' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBSTRESC' を '1' に更

In [20]:
# 更新データ確認
compare_data(dataset_list, dataset_list_updated)

with open("dataset_list_updated.json", "w") as f:
  json.dump(dataset_list_updated, f)

--- ItemGroupOID: AE ---
USUBJID = 01-701-1015, AESEQ = 3:
  Updated:
    AEENDTC: '2014-01-11' -> '2014-01-09'
    AEENDY: 10 -> 8
    AESER: 'N' -> 'Y'
    AESHOSP: 'N' -> 'Y'
    AESTDTC: '2014-01-09' -> '2014-01-11'
    AESTDY: 8 -> 10

USUBJID = 01-701-1028, AESEQ = 1:
  Updated:
    AESTDTC: '2013-07-21' -> '2013-07-01'
    AESTDY: 3 -> -17
    AETERM: 'APPLICATION SITE ERYTHEMA' -> "PARKINSON'S DISEASE"

USUBJID = 01-701-1034, AESEQ = 2:
  Updated:
    AETERM: 'FATIGUE' -> 'MALIGNANT HYPERTENSION'

USUBJID = 01-701-1047, AESEQ = 3:
  Updated:
    AEENDTC: '' -> '2013-03-05'
    AEENDY: None -> 22

USUBJID = 01-701-1047, AESEQ = 4:
  Updated:
    AETERM: 'BUNDLE BRANCH BLOCK LEFT' -> 'HYPERTENSION'

USUBJID = 01-701-1153, AESEQ = 2:
  Updated:
    AEACN: '' -> 'DRUG WITHDRAWN'

USUBJID = 01-701-1180, AESEQ = 6:
  Updated:
    AETERM: 'MICTURITION URGENCY' -> 'SUDDEN DEATH'

USUBJID = 01-701-1363, AESEQ = 1:
  Updated:
    AEENDTC: '2013-06-15' -> '2013-06-14'
    AEENDY: 17 -> 16

# LLMへの送信

In [21]:
!pip install sseclient-py
import requests
import sseclient
from IPython.display import display, Markdown

from google.colab import userdata
api_key = userdata.get('Dify_DatasetJSON')
user_id = 'JPMA_Sample'

## 関数定義

In [159]:
import json
import time
import requests
import sseclient

# 定数
DIFY_API_URL = 'https://api.dify.ai/v1/workflows/run'
CONTENT_TYPE_JSON = 'application/json'

def call_dify_api(api_key: str, payload: dict, stream: bool = False) -> requests.Response:
    """Dify APIを呼び出す共通関数"""
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': CONTENT_TYPE_JSON
    }
    try:
        response = requests.post(DIFY_API_URL, headers=headers, json=payload, stream=stream)
        response.raise_for_status()  # HTTPエラーが発生した場合に例外を発生させる
        return response
    except requests.exceptions.RequestException as e:
        print(f"API呼び出しエラー: {e}")
        raise

def run_dify_workflow(api_key: str, workflow_inputs: dict, user_id: str, streaming: bool = False) -> dict | sseclient.Event:
    """Difyワークフローを実行する

    Args:
        api_key: Dify APIキー
        workflow_inputs: ワークフローへの入力
        user_id: ユーザーID
        streaming: ストリーミングモードで実行するかどうか (Falseの場合はブロッキングモード)

    Returns:
        ストリーミングモードの場合はsseclient.Eventのイテレータ、
        ブロッキングモードの場合はAPIのレスポンスのJSONを辞書型で返す
    """
    payload = {
        'inputs': workflow_inputs,
        'response_mode': 'streaming' if streaming else 'blocking',
        'user': user_id
    }
    response = call_dify_api(api_key, payload, stream=streaming)
    if streaming:
        client = sseclient.SSEClient(response)
        return client.events()
    else:
        return response.json()

def safe_print_event_data(event: sseclient.Event):
    """
    与えられたSSEイベントデータから、存在する場合に特定の値を出力します。
    キーが存在しない場合は何も出力しません。Statusが存在する場合にのみTitleと結合させて表示します。

    Args:
        event: イベントデータを含むSSEイベントオブジェクト。event.data属性がJSON文字列であることを想定。
    """
    try:
        data = json.loads(event.data)

        if 'event' in data:
            print(f"Event: {data['event']}")

        if 'data' in data:
            title = data['data'].get('title')
            status = data['data'].get('status')
            if title is not None and status is not None:
                print(f"Node: {title} ({status})")
            elif title is not None:
                print(f"Node: {title}") # Statusが存在しない場合はTitleのみ表示

            if 'error' in data['data']:
                print(f"Error: {data['data']['error']}")
            if 'elapsed_time' in data['data']:
                print(f"Elapsed time: {data['data'].get('elapsed_time')}")
            if 'total_tokens' in data['data']:
                print(f"Total tokens: {data['data'].get('total_tokens')}")

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON event data: {e}")
    except AttributeError as e:
        print(f"Error accessing event data attribute: {e}")

def run_workflow_with_retry(api_key: str, workflow_inputs: dict, user_id: str, max_retries: int = 3, retry_delay: int = 20):
    """ワークフローを実行し、エラー発生時にリトライを行う (ストリーミングモード専用)"""
    for retry in range(max_retries + 1):
        print(f"--- 試行回数: {retry + 1} ---")
        success = True
        try:
            for event in run_dify_workflow(api_key, workflow_inputs, user_id, streaming=True):
                safe_print_event_data(event)
                try:
                    event_data = json.loads(event.data)
                    if event_data.get('data', {}).get('error') is not None:
                        print(f"エラーが検出されました: {event_data['data']['error']}")
                        success = False
                        break
                except json.JSONDecodeError:
                    print("JSONデコードエラーが発生しました。")
                    success = False
                    break
                print('------')

            if success:
                print("ワークフローが正常に完了しました。")
                return json.loads(event.data)
            elif retry < max_retries:
                print(f"エラーが発生したため、{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

        except requests.exceptions.RequestException as e:
            print(f"APIリクエスト中にエラーが発生しました: {e}")
            success = False
            if retry < max_retries:
                print(f"{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

## 出力フォーマットの定義

In [160]:
# Markdown
Markdown_General = '''
**出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。
'''
Taks1_Markdown = Markdown_General+'''
    1. 症例サマリー：[USUBJID]
        *   YYYY年MM月DD日 (Day XX): [有害事象、検査値、バイタルサインなどのイベントを、異常所見を中心に簡潔な文章で記載]

    2. 疑義事項: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]
'''

Taks2_Markdown = Markdown_General+'''
    1. 確認した症例：[USUBJID]

    2. 医療機関に問い合わせるクエリ: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

    3. 医療機関に問い合わせない疑義事項: [あり/なし]
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **疑義事項:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]
'''

Taks3_Markdown = Markdown_General+'''
    1. 確認した症例：[USUBJID]

    2. プロトコル逸脱: [あり/なし]
        *   **逸脱No.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **逸脱内容:** [具体的な逸脱内容を簡潔に記述。例：被験者XXXは、プロトコルで規定された投与量を超える量の治験薬を投与された]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

    3. 医療機関に問い合わせるクエリ: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

'''

# JSON
JSON_General = '''
**出力形式:**

*   すべての出力は指定されるJSON Schemaを用いて出力してください。
*   コードブロックや改行コードは使用せず、"{"で開始し、"}"で終わるJSONオブジェクト形式で出力してください。

**出力言語:**

*   JSONのValueは日本語で出力します

**JSON Schema**
'''

Task1_JSON = JSON_General+'''
{ "name": "Clinical_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "timeline": { "type": ["array", "null"], "description": "有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリー", "items": { "type": "object", "properties": { "date": { "type": "string", "description": "Date of the event.  Allows full (YYYY-MM-DD), partial (YYYY-MM), or year-only (YYYY) formats." }, "day": { "type": "integer", "description": "Day relative to study start (Day 1).  Allows negative values for pre-treatment days." }, "details": { "type": "string", "description": "異常所見を中心に簡潔な文章で記載する。正常範囲内の変動は省略可能。" } }, "required": [ "date", "day", "details" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "timeline", "queries" ], "additionalProperties": false } }
'''

Task2_JSON = JSON_General+'''
{ "name": "_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "data_issues": { "type": ["array", "null"], "description": "List of data issues identified during review", "items": { "type": "object", "properties": { "issue_no": { "type": "integer", "description": "Unique issue number" }, "variables": { "type": "array", "description": "List of variables and their values related to the issue", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } }, "inconsistency": { "type": "string", "description": "具体的な矛盾の内容を記述" }, "cause": { "type": "string", "description": "問題点の原因（推測）" }, "resolution": { "type": "string", "description": "対応策（提案）" } }, "required": [ "issue_no", "variables", "inconsistency", "cause", "resolution" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "data_issues", "queries" ], "additionalProperties": false } }
'''

Task3_JSON = JSON_General+'''
{ "name": "Protocol_Deviation_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "deviations": { "type": ["array", "null"], "description": "List of protocol deviations", "items": { "type": "object", "properties": { "deviation_no": { "type": "integer", "description": "Unique deviation number" }, "impact": { "type": "string", "description": "Impact of the deviation on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "variables": { "type": "array", "description": "List of variables and their values related to the deviation", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } }, "description": { "type": "string", "description": "具体的な逸脱内容を簡潔に記述。" }, "protocol_reference": { "type": "string", "description": "プロトコルの該当するセクション、ページ番号などを記載" }, "justification": { "type": "string", "description": "判断理由" } }, "required": [ "deviation_no", "impact", "variables", "description", "protocol_reference", "justification" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "deviations", "queries" ], "additionalProperties": false } }
'''


## プロンプトの作成

In [161]:
with open('define_xml/define.xml', 'r') as f:
  define_xml = f.read()


SysPrompt = '''
あなたは、臨床試験データのレビューを支援するAIアシスタントです。以下の前提知識を理解した上で、ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援してください。各タスクでは、ユーザープロンプトで指定された役割になりきって回答してください。

**前提知識:**

*   臨床試験においては患者の安全性が最優先され、有害事象の評価は特に重要です。
*   SDTM (Study Data Tabulation Model) は、CDISCによって策定された臨床試験データの標準モデルです。
*   Define.xmlはSDTMデータの構造を記述したメタデータファイルであり、参考情報として使用します。JSONデータ自体の内容、医学的妥当性、プロトコルとの整合性を優先してレビューしてください。
*   SDTMデータは、DM、AE、VS、LBなど、複数のドメイン（データセット）に分かれています。
*   報告されるJSONデータには、データ入力時の間違いが含まれる可能性があります。
*   提供された情報のみに基づいて回答を作成してください。想像やハルシネーションに基づいた回答は作成してはいけません。

**その他:**

*   指定された出力フォーマットに厳密に従って出力してください。
*   JSONデータまたはDefine.xmlの形式が不正な場合は、その旨をエラーメッセージとして出力してください。
'''




UserInput_Task1 = '''
あなたは臨床試験の専門医です。以下の指示に従い、提供される情報（プロトコル、JSONデータ、Define.xml）を基に、臨床試験データのレビューとクエリ作成（必要な場合）を行ってください。

**1. 症例サマリーの作成:**

*   **参照情報:** JSONデータ、Define.xml
*   **タスク:**
    *   JSONデータとDefine.xmlを参照し、有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリーを作成してください。
    *   特に、**異常所見**を中心に簡潔な文章で記載してください。正常範囲内の変動は省略して構いません。
    *   各イベントの日時は、Define.xmlに定義された日付変数などを参考に、正確に特定してください。

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   以下のJSONデータのレビュー観点に基づき、JSONデータを改めて点検してください。
    *   医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *  **疑義事項がない場合は、クエリを作成する必要はありません。**「疑義事項なし」と回答してください。

*   **JSONデータのレビュー観点 (これらに限定されない):**
    *   **安全性:** 有害事象(AEドメイン)の報告内容は、医学的に妥当であるか？
    *   **医学的妥当性:** 検査値(LBドメイン)の変動、バイタルサイン(VSドメイン)の変動、併用薬(CMドメイン)との相互作用など、時間経過とともに医学的に問題となる点は見られるか？
    *   **有効性:** 特定された主要評価項目および副次評価項目について、その時間的変化は期待される効果と一致しているか？
    *   **その他:** 患者背景(DMドメイン)、既往歴(MHドメイン)、有害事象(AEドメイン)、治療歴(EXドメイン, CMドメイン)などを総合的に考慮し、時間経過を加味して安全性に懸念を生じる事項があれば記載してください。
    *   **プロトコル逸脱 (疑い):** 選択/除外基準、投与量、併用禁止薬、評価スケジュール、有害事象報告などについて、プロトコルからの逸脱の疑いがないか確認してください。（関連ドメイン: DM, MH, EX, CM, LB, VS, AEなど）
'''



UserInput_Task2 = '''
あなたはクリニカルデータマネージャーです。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、データ整合性レビューとクエリ作成（必要な場合）を行ってください。

**1. データ整合性レビュー:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、データの不整合が疑われる問題点を検出してください。
    *   **特に、以下の点に焦点を当ててレビューしてください。**
        *   **クロスドメイン整合性:** 異なるSDTMドメイン間で、データに矛盾がないか、ドメイン間の関連性が正しく表現されているか。
            *   **具体的な確認例 (これらに限定されない):**
                *   DM.SEXとAEにおける妊娠関連の有害事象
                *   AEの有害事象発現日や治験薬との関連性と、EXの治験薬の投与期間
                *   LBの検査値異常とAEの関連有害事象
                *   VSのバイタルサイン異常とAEの関連有害事象
                *   CM.CMTRTとAE/MHで報告されている疾患・既往歴との矛盾

        *   **単一ドメイン内の整合性:** Define.xmlの定義に照らして、矛盾なく解釈できるデータになっているか、プロトコルに照らしてデータの関連性が正しく表現されているか。
        *   **異常値:** Define.xmlで定義された範囲外、または医学的にありえない値がないか。
        *   **欠損値:** 欠損値の有無と理由（推測できる場合）。多い場合は原因を推測。
        *   **プロトコル逸脱 (データ品質の観点から):** データ入力/収集で、プロトコルからの逸脱（例：必須項目の未入力、不適切な時期のデータ収集）がないか。

    *   Define.xmlとデータの間に不整合がある場合は、「Define.xmlの修正候補」として報告してください。

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   データ整合性レビューの結果、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **疑義事項がない場合は、クエリを作成する必要はありません。**
'''



UserInput_Task3 = '''
あなたは、臨床試験の専門医、データマネージャー、CRAの視点を持つ、プロトコル遵守状況の確認者です。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、プロトコル逸脱の検出とクエリ作成（必要な場合）を行ってください。

**1. プロトコル逸脱の検出:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、プロトコルからの逸脱を検出してください。
    *   Define.xmlは参考情報として活用し、データとプロトコルの内容を比較して逸脱を判断してください。
    *   **検出対象とすべき主要なプロトコル逸脱の例 (これらに限定されない):**
        *   **選択/除外基準違反:** (関連SDTMドメイン: DM, MH など)
        *   **投与量違反:** (関連SDTMドメイン: EX)
        *   **併用禁止薬の使用:** (関連SDTMドメイン: CM)
        *   **評価スケジュール違反:** (関連SDTMドメイン: LB, VS, その他)
        *   **有害事象報告違反**: (関連SDTMドメイン: AE)

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   プロトコル逸脱を判定するために、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、プロトコル逸脱が臨床試験の評価項目に与える影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **プロトコル逸脱に関する疑義事項がない場合は、クエリを作成する必要はありません。**
'''


UserInput_end1 = '''\n---\n\n**データ:**\n\n*   臨床試験データ（JSON形式、SDTM準拠）:\n\n```json\n'''
UserInput_end2 = '''\n```\n\n*   データ定義ファイル（Define.xml）:\n\n```xml\n''' + define_xml + '''```\n'''

In [162]:
def create_workflow_input(ModelName, SysPrompt, UserInput_Task, datasetjson, UserInput_end1, UserInput_end2):
    return {
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_Task + UserInput_end1 + datasetjson + UserInput_end2,
        'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}
    }

## 実行

In [163]:
# ModelNameの設定
#ModelName = 'gemini-2.0-flash'
#ModelName = 'gemini-2.0-flash-exp'
#ModelName = 'gemini-2.0-flash-exp-multi'
ModelName = 'gemini-2.0-pro-exp'
#ModelName = 'gemini-2.0-pro-exp-02-05'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21-multi'
#ModelName = 'gemini-2.0-flash-thinking-exp'
#ModelName = 'gemini-2.0-flash-thinking-exp-multi'


# データ更新症例の抽出
updated_subjects = []
for l in Target_data:
  updated_subjects.append(l[1])

updated_subjects = sorted(list(set(updated_subjects)))
print(updated_subjects)
print(len(updated_subjects))


['01-701-1015', '01-701-1023', '01-701-1028', '01-701-1034', '01-701-1047', '01-701-1097', '01-701-1111', '01-701-1118', '01-701-1146', '01-701-1148', '01-701-1153', '01-701-1180', '01-701-1181', '01-701-1363', '01-701-1383', '01-701-1387', '01-702-1082', '01-703-1042', '01-703-1076', '01-703-1086', '01-703-1096', '01-703-1258', '01-703-1279', '01-703-1299', '01-703-1335', '01-703-1403', '01-704-1008', '01-704-1009', '01-704-1010', '01-704-1017']
30


In [164]:
# output mode: Markdown / JSON
Output_Format = "Markdown"

if Output_Format == "Markdown":
  UserInput_Task1 = UserInput_Task1 + Taks1_Markdown
  UserInput_Task2 = UserInput_Task2 + Taks2_Markdown
  UserInput_Task3 = UserInput_Task3 + Taks3_Markdown
elif Output_Format == "JSON":
  UserInput_Task1 = UserInput_Task1 + Task1_JSON
  UserInput_Task2 = UserInput_Task2 + Task2_JSON
  UserInput_Task3 = UserInput_Task3 + Task3_JSON


In [165]:
import pandas as pd

results_list = []

for subj in updated_subjects[0:1]:
    datasetjson = filter_data(dataset_list_updated, subj)
    print(f"処理完了：'datasetjson' に USUBJID が {subj} のデータを出力しました。")

    row_data = {'Subject': subj}  # 各行のデータを格納する辞書

    # Task 1 の処理
    workflow_inputs_Task1 = create_workflow_input(ModelName, SysPrompt, UserInput_Task1, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
    try:
        result_Task1 = run_workflow_with_retry(api_key, workflow_inputs_Task1, user_id)
        output_Task1 = result_Task1['data']['outputs']['text']
        display(Markdown(output_Task1))
        row_data['Task1'] = output_Task1
    except Exception as e:
        print(f"Task 1 でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task1'] = "Error"

    # Task 2 の処理
    workflow_inputs_Task2 = create_workflow_input(ModelName, SysPrompt, UserInput_Task2, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
    try:
        result_Task2 = run_workflow_with_retry(api_key, workflow_inputs_Task2, user_id)
        output_Task2 = result_Task2['data']['outputs']['text']
        display(Markdown(output_Task2))
        row_data['Task2'] = output_Task2
    except Exception as e:
        print(f"Task 2 でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task2'] = "Error"

    # Task 3 の処理
    workflow_inputs_Task3 = create_workflow_input(ModelName, SysPrompt, UserInput_Task3, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
    try:
        result_Task3 = run_workflow_with_retry(api_key, workflow_inputs_Task3, user_id)
        output_Task3 = result_Task3['data']['outputs']['text']
        display(Markdown(output_Task3))
        row_data['Task3'] = output_Task3
    except Exception as e:
        print(f"Task 3 でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task3'] = "Error"

    results_list.append(row_data)

# DataFrameを作成
df_results = pd.DataFrame(results_list)

# DataFrameを表示
display(df_results)

警告：データセット 'CDISCPILOT01.te' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ts' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ti' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.tv' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ta' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
処理完了：'datasetjson' に USUBJID が 01-701-1015 のデータを出力しました。
--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 1.603271
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 1.712379
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.372468
------
Event: node_started
Node: gemini-2.0-pro-exp
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chun

1. 症例サマリー：01-701-1015
    *   2013年12月26日 (Day -7): スクリーニング検査にて、アルカリホスファターゼ(ALP)低値(34 U/L, 基準値35-115)、アニソサイトーシス(Anisocytes)異常(Grade 1)、アスパラギン酸アミノトランスフェラーゼ(AST)高値(40 U/L, 基準値9-34)を認めた。身長147.32cm、体重53.98kg。脈拍は臥位57回/分、立位62-65回/分と徐脈傾向。血圧は臥位131/64 mmHg、立位129-147/57-83 mmHg。教育歴16年。アルツハイマー病と診断 (2010年4月30日発症)。既往歴に動悸、頭痛、耳鳴、胸やけ、脚のしびれ、子宮部分摘出術(1986年)、胆石(2012年)、甲状腺部分切除術(1973年)、咽頭痛(2013年12月)、扁桃摘出術(1973年)あり。併用薬としてアスピリン(2003年～)、カルシウム(2013年～)、プレマリン(2006年～)、タイレノール(2003年～)を使用中。MMSE 23点 (Day -7, Visit 1)。
    *   2013年12月31日 (Day -2): スクリーニング検査。脈拍は臥位56回/分、立位57-60回/分。血圧は臥位138/68 mmHg、立位137-145/59-71 mmHg。
    *   2014年01月02日 (Day 1): ベースライン。プラセボ投与開始。体重54.43kg。脈拍は臥位56回/分、立位59回/分。血圧は臥位130/56 mmHg、立位121-131/51-61 mmHg。ADAS-Cog(11)合計点13点。NPI-X合計点0点。DADスコア全項目「Yes」。
    *   2014年01月03日 (Day 2): 投与部位紅斑(軽度)、投与部位掻痒(軽度)発現。治験薬との関連あり。併用薬としてネオスポリン（外用）開始。
    *   2014年01月11日 (Day 10): 下痢(軽度)発現。本有害事象は重篤（入院）と判断された。
    *   2014年01月16日 (Day 15): Week 2 Visit。アラニンアミノトランスフェラーゼ(ALT)高値(41 U/L, 基準値6-34)を認めた。ASTは正常化(33 U/L)。体重53.07kg。脈拍は臥位58回/分、立位61-62回/分。血圧は臥位114/56 mmHg、立位121-132/50-54 mmHg。NPI-X合計点0点。
    *   2014年01月30日 (Day 29): Week 4 Visit。赤血球平均容積(MCV)低値(78 fL, 基準値80-100)を認めた。ALTは正常化(18 U/L)。体重53.98kg。脈拍は臥位59回/分、立位59-62回/分。血圧は臥位138/64 mmHg、立位132-137/53-55 mmHg。NPI-X合計点0点。
    *   2014年02月12日 (Day 42): Week 6 Visit。体重53.07kg。脈拍は臥位55回/分、立位56-57回/分。血圧は臥位148/55 mmHg、立位137-138/60-63 mmHg。NPI-X合計点0点。
    *   2014年03月05日 (Day 63): Week 8 Visit。体重53.07kg。脈拍は臥位57回/分、立位57-60回/分。血圧は臥位138/67 mmHg、立位140-146/62-71 mmHg。ADAS-Cog(11)合計点8点。CIBIC+評価「No Change」。DADスコア全項目「Yes」。NPI-X合計点0点。
    *   2014年03月26日 (Day 84): Week 12 Visit。体重53.07kg。脈拍は臥位54回/分、立位52-53回/分。血圧は臥位139/68 mmHg、立位138-146/60-64 mmHg。併用薬としてヒドロコルチゾン（外用）開始(Day 85)。NPI-X合計点0点。
    *   2014年05月07日 (Day 126): Week 16 Visit。MCV低値(79 fL, 基準値80-100)を認めた。体重53.07kg。脈拍は臥位60回/分、立位55-61回/分。血圧は臥位163/66 mmHg、立位145-152/66-68 mmHg。ADAS-Cog(11)合計点11点。CIBIC+評価「No Change」。DADスコア全項目「Yes」。NPI-X合計点0点。
    *   2014年05月21日 (Day 140): Week 20 Visit。体重53.07kg。脈拍は臥位54回/分、立位58-59回/分。血圧は臥位137/67 mmHg、立位136-139/63-65 mmHg。NPI-X合計点0点。
    *   2014年06月18日 (Day 168): Week 24 Visit。尿比重低値(1.005, 基準値1.006-1.03)を認めた。体重53.07kg。脈拍は臥位55回/分、立位56-57回/分。血圧は臥位129/63 mmHg、立位131-137/57-71 mmHg。ADAS-Cog(11)合計点8点。CIBIC+評価「No Change」。DADスコア全項目「Yes」。NPI-X合計点0点。
    *   2014年07月02日 (Day 182): Week 26 Visit (試験終了)。体重53.52kg。脈拍は臥位60回/分、立位59-61回/分。血圧は臥位127/61 mmHg、立位128-129/55-59 mmHg。NPI-X合計点0点。投与部位紅斑および掻痒は未回復のまま試験終了。下痢は回復。

2.  疑義事項: あり
    *   **クエリNo.:** 1
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 有害事象の下痢 (AESEQ=3) について、開始日 (AESTDTC) が2014-01-11、終了日 (AEENDTC) が2014-01-09と報告されており、日付が矛盾しています。正しい日付をご確認ください。また、本有害事象は重篤 (入院) であったにも関わらず、重症度が軽度 (MILD)、治験薬との関連性が REMOTE と評価されています。評価の妥当性について再確認し、必要であれば修正をお願いします。
        *   **判断理由:** 有害事象の日付の正確性は安全性評価において重要である。また、重篤な有害事象の重症度と関連性の評価は、治験薬の安全性プロファイルを正しく理解するために不可欠である。入院を要する下痢が軽度かつ治験薬との関連性が低いという評価には医学的な疑問がある。
        *   **判断根拠:**
            *   AE.USUBJID = "01-701-1015"; AE.AESEQ = 3; AE.AETERM = "DIARRHOEA"; AE.AESTDTC = "2014-01-11"; AE.AEENDTC = "2014-01-09"; AE.AESER = "Y"; AE.AESHOSP = "Y"; AE.AESEV = "MILD"; AE.AEREL = "REMOTE"

    *   **クエリNo.:** 2
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 有害事象の投与部位紅斑 (AESEQ=1) および投与部位掻痒 (AESEQ=2) が、試験終了時点 (2014-07-02) で未回復 (NOT RECOVERED/NOT RESOLVED) と報告されています。プロトコルでは、発疹がみられた被験者に対して最終試験来院の2週間後に Rash followup visit (Visit 501) を実施する規定がありますが、本症例では実施記録が見当たりません。Visit 501 が実施されなかった理由、およびこれらの有害事象の最終的な転帰について確認してください。
        *   **判断理由:** 有害事象の転帰確認は安全性の評価に不可欠であり、プロトコルで規定されたフォローアップが実施されていない場合はその理由を確認する必要がある。
        *   **判断根拠:**
            *   AE.USUBJID = "01-701-1015"; AE.AESEQ = 1; AE.AETERM = "APPLICATION SITE ERYTHEMA"; AE.AEOUT = "NOT RECOVERED/NOT RESOLVED"; AE.AEENDTC = ""
            *   AE.USUBJID = "01-701-1015"; AE.AESEQ = 2; AE.AETERM = "APPLICATION SITE PRURITUS"; AE.AEOUT = "NOT RECOVERED/NOT RESOLVED"; AE.AEENDTC = ""
            *   DS.USUBJID = "01-701-1015"; DS.DSDECOD = "COMPLETED"; DS.DSSTDTC = "2014-07-02"
            *   TV.VISITNUM = 501; TV.VISIT = "Rash followup"; TV.TVSTRL = "If subject experienced rash, then Rash Followup occurs at End of last study visit + 2W"
            *   SVデータに VISITNUM = 501 の記録なし

    *   **クエリNo.:** 3
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** プロトコル除外基準 [28b] では、血清中葉酸 (Folate) 値が基準値下限未満の場合、除外することになっていますが、本症例のスクリーニング時 (Visit 1) の葉酸データが見当たりません。葉酸が測定されなかった理由、または測定結果についてご確認ください。
        *   **判断理由:** 被験者の適格性を確認するために必要な検査データが欠損している。プロトコル逸脱の可能性を排除できない。
        *   **判断根拠:**
            *   LB.USUBJID = "01-701-1015"; LB.VISITNUM = 1 (SCREENING 1); LB.LBTESTCD に "FOL" (Folate) が含まれていない。
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [28b]: "Central laboratory test values below reference range for folate..."

    *   **クエリNo.:** 4
        *   **臨床試験結果への影響度合い:** Minor
        *   **医療機関への問い合わせ文面:** 併用薬として報告されている "NEOSPORIN /USA/" (CMSEQ=17 等) および "TYLENOL" (CMSEQ=2 等) について、標準化された薬剤名 (CMDECOD) が "UNCODED" となっています。可能であればWHODrug等を用いてコーディングを実施してください。
        *   **判断理由:** データ標準化のため、可能な限り薬剤名のコーディングが望ましい。
        *   **判断根拠:**
            *   CM.USUBJID = "01-701-1015"; CM.CMTRT = "NEOSPORIN /USA/"; CM.CMDECOD = "UNCODED"
            *   CM.USUBJID = "01-701-1015"; CM.CMTRT = "TYLENOL"; CM.CMDECOD = "UNCODED"

    *   **クエリNo.:** 5
        *   **臨床試験結果への影響度合い:** Minor
        *   **医療機関への問い合わせ文面:** 併用薬 HYDROCORTISONE (CMSEQ=48 等) は投与経路 (CMROUTE) が TOPICAL となっていますが、薬剤クラス (CMCLAS) が "SYSTEMIC HORMONAL PREPARATIONS, EXCL." と報告されています。局所用製剤の場合、クラス分類が適切かご確認ください。
        *   **判断理由:** データ精度向上のため、適切な薬剤クラス分類が望ましい。
        *   **判断根拠:**
            *   CM.USUBJID = "01-701-1015"; CM.CMSEQ = 48; CM.CMTRT = "HYDROCORTISONE"; CM.CMROUTE = "TOPICAL"; CM.CMCLAS = "SYSTEMIC HORMONAL PREPARATIONS, EXCL."

--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.056368
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.255455
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.055899
------
Event: node_started
Node: gemini-2.0-pro-exp
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk

1. 確認した症例：01-701-1015

2. 医療機関に問い合わせるクエリ: あり
    *   **クエリNo.:** 1
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 有害事象「DIARRHOEA」(AESEQ=3)について、有害事象発現日(AESTDTC)が2014-01-11、有害事象消失日(AEENDTC)が2014-01-09と報告されています。消失日が発現日より前になっていますのでご確認ください。
        *   **判断理由:** 有害事象の期間が正しく評価できず、安全性評価に影響を与えるため。
        *   **判断根拠:**
            * AE.USUBJID = 01-701-1015; AE.AESEQ = 3; AE.AETERM = DIARRHOEA; AE.AESTDTC = 2014-01-11; AE.AEENDTC = 2014-01-09

    *   **クエリNo.:** 2
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 併用薬「NEOSPORIN /USA/」(CMSEQ=17, 22, 27, 32, 37, 42, 47, 53, 59, 65)について、薬剤コードが付与されていません(CMDECOD=UNCODED)。正式な薬剤名およびWHO Drug Dictionaryコードをご確認ください。また、可能であれば使用理由（Indication）もご教示ください。
        *   **判断理由:** 薬剤名が特定できず、併用禁止薬の確認や相互作用の評価、および有害事象との関連性評価に影響を与える可能性があるため。
        *   **判断根拠:**
            * CM.USUBJID = 01-701-1015; CM.CMTRT = NEOSPORIN /USA/; CM.CMDECOD = UNCODED; CM.CMINDC =

3. 医療機関に問い合わせない疑義事項: あり
    *   **臨床試験結果への影響度合い:** Minor
        *   **疑義事項:** 併用薬「HYDROCORTISONE」(CMSEQ=48, 54, 60, 66) の使用理由(Indication)が入力されていません。
        *   **判断理由:** AEドメインで報告されている「APPLICATION SITE ERYTHEMA」(AESEQ=1) および「APPLICATION SITE PRURITUS」(AESEQ=2)（いずれも発現日 2014-01-03、転帰 未回復/未解決）の治療に使用されたと推測されますが、明記されていません。Indicationが入力されることが望ましいですが、使用開始日（2014-03-27）とAEの発現時期から、AE治療目的である可能性が高いと判断し、現時点での問い合わせは不要としました。
        *   **判断根拠:**
            * CM.USUBJID = 01-701-1015; CM.CMTRT = HYDROCORTISONE; CM.CMINDC = ; CM.CMSTDTC = 2014-03-27
            * AE.USUBJID = 01-701-1015; AE.AESEQ = 1; AE.AETERM = APPLICATION SITE ERYTHEMA; AE.AESTDTC = 2014-01-03; AE.AEOUT = NOT RECOVERED/NOT RESOLVED
            * AE.USUBJID = 01-701-1015; AE.AESEQ = 2; AE.AETERM = APPLICATION SITE PRURITUS; AE.AESTDTC = 2014-01-03; AE.AEOUT = NOT RECOVERED/NOT RESOLVED

    *   **臨床試験結果への影響度合い:** Minor
        *   **疑義事項:** Week 2 (VISITNUM=4) の臨床検査において、ALT (Alanine Aminotransferase) が基準値上限 (34 U/L) を超える値 (41 U/L) を示しています (LBSEQ=41)。
        *   **判断理由:** 軽度の上昇であり、プロトコルで規定された処置基準（例: >3xULN など）には該当しない可能性が高いです。また、関連する有害事象も報告されていません。後続のVisit (Week 4以降) では基準値内に回復しているため、現時点での問い合わせは不要と判断しました。ただし、最終的な評価のため、結果は注視します。
        *   **判断根拠:**
            * LB.USUBJID = 01-701-1015; LB.LBTESTCD = ALT; LB.VISITNUM = 4; LB.LBSEQ = 41; LB.LBORRES = 41; LB.LBORNRHI = 34; LB.LBNRIND = HIGH
            * LB.USUBJID = 01-701-1015; LB.LBTESTCD = ALT; LB.VISITNUM = 5; LB.LBSEQ = 76; LB.LBORRES = 18; LB.LBNRIND = NORMAL

    *   **臨床試験結果への影響度合い:** Minor
        *   **疑義事項:** Week 16 (VISITNUM=10) のバイタルサインにおいて、収縮期血圧 (Systolic Blood Pressure, SUPINE) が 163 mmHg と高値を示しています (VSSEQ=116)。
        *   **判断理由:** プロトコルの除外基準 [17] f) Uncontrolled hypertension に抵触しないか確認が必要です。ベースライン (VISITNUM=3, VSSEQ=92) では130 mmHg ですが、他のVisitでも140 mmHgを超える測定値が散見されます。関連する有害事象は報告されていません。単回の測定値であり、他の測定値も考慮すると直ちに重大な安全性懸念とは判断し難いため、現時点での問い合わせは保留とします。ただし、最終的な評価のため、結果は注視します。
        *   **判断根拠:**
            * VS.USUBJID = 01-701-1015; VS.VSTESTCD = SYSBP; VS.VSPOS = SUPINE; VS.VISITNUM = 10; VS.VSSEQ = 116; VS.VSORRES = 163
            * VS.USUBJID = 01-701-1015; VS.VSTESTCD = SYSBP; VS.VSPOS = SUPINE; VS.VISITNUM = 3; VS.VSSEQ = 92; VS.VSORRES = 130
            * Protocol Section 3.4.2.2 Exclusion Criteria [17] f) Uncontrolled hypertension.

--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.047253
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.223096
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.085516
------
Event: node_started
Node: gemini-2.0-pro-exp
------
Event: node_finished
Node: gemini-2.0-pro-exp (failed)
Error: [google] Error: PluginInvokeError: {"args":null,"error_type":"PluginDaemonInnerError","message":"encountered an error: invalid character '\u003c' looking for beginning of value status: 504 Gateway Time-out original response: \u003c!DOCTYPE html\u003e"}
Elapsed time: 60.220668
エラーが検出されました: [google] Error: PluginInvokeError: {"args":null,"error_type":"PluginDaemonInnerError","message":"encountered an error: invalid character '\u003c' looking for beginni

1. 確認した症例：01-701-1015

2. プロトコル逸脱: あり
    *   **逸脱No.:** 1
        *   **臨床試験結果への影響度合い:** Major
        *   **逸脱内容:** 被験者 01-701-1015 は、スクリーニング時（Visit 1）の臨床検査において、除外基準[27b]に抵触するAST(SGOT)高値を示したにも関わらず、試験に登録された。
        *   **プロトコル該当箇所:** Section 3.4.2.2 Exclusion Criteria [27b] (Page 15)
        *   **判断理由:** スクリーニング時のAST値が基準値上限を超えているため。
        *   **判断根拠:**
            *   LB.USUBJID = 01-701-1015
            *   LB.VISITNUM = 1
            *   LB.LBTESTCD = AST
            *   LB.LBORRES = 40
            *   LB.LBORRESU = U/L
            *   LB.LBSTRESN = 40
            *   LB.LBSTNRHI = 34

    *   **逸脱No.:** 2
        *   **臨床試験結果への影響度合い:** Minor
        *   **逸脱内容:** 被験者 01-701-1015 は、スクリーニング時（Visit 1）の臨床検査において、除外基準[27b]に抵触するALP低値を示したにも関わらず、試験に登録された。
        *   **プロトコル該当箇所:** Section 3.4.2.2 Exclusion Criteria [27b] (Page 15)
        *   **判断理由:** スクリーニング時のALP値が基準値下限を下回っているため。
        *   **判断根拠:**
            *   LB.USUBJID = 01-701-1015
            *   LB.VISITNUM = 1
            *   LB.LBTESTCD = ALP
            *   LB.LBORRES = 34
            *   LB.LBORRESU = U/L
            *   LB.LBSTRESN = 34
            *   LB.LBSTNRLO = 35

    *   **逸脱No.:** 3
        *   **臨床試験結果への影響度合い:** Major
        *   **逸脱内容:** 被験者 01-701-1015 の Ambulatory ECG の実施時期（装着: Visit 3.5/Day 13, 除去: Visit 6/Day 30）が、プロトコルで規定された時期（装着: Visit 2/Day -1, 除去: Visit 3/Day 1）と異なっていた。
        *   **プロトコル該当箇所:** Section 3.1 Summary of Study Design (Page 8), Protocol Attachment LZZT.1 Schedule of Events (Page 53)
        *   **判断理由:** SVデータおよびVSデータから確認されるAmbulatory ECG関連のVisit（3.5, 6）と実施日（Day 13, Day 31）が、プロトコルおよびスケジュール表の記載（Visit 2, 3）と一致しないため。
        *   **判断根拠:**
            *   SV.USUBJID = 01-701-1015
            *   SV.VISIT = AMBUL ECG PLACEMENT; SV.VISITNUM = 3.5; SV.VISITDY = 13; SV.SVSTDTC = 2014-01-14
            *   SV.VISIT = AMBUL ECG REMOVAL; SV.VISITNUM = 6; SV.VISITDY = 30; SV.SVSTDTC = 2014-02-01
            *   VS.USUBJID = 01-701-1015
            *   VS.VISIT = AMBUL ECG PLACEMENT; VS.VISITNUM = 3.5; VS.VSDY = 13
            *   VS.VISIT = AMBUL ECG REMOVAL; VS.VISITNUM = 6; VS.VSDY = 31
            *   TV.VISIT = SCREENING 2; TV.VISITNUM = 2; TV.VISITDY = -1 (プロトコル上のAmbulatory ECG装着Visit)
            *   TV.VISIT = BASELINE; TV.VISITNUM = 3; TV.VISITDY = 1 (プロトコル上のAmbulatory ECG除去Visit)

    *   **逸脱No.:** 4
        *   **臨床試験結果への影響度合い:** Minor
        *   **逸脱内容:** 被験者 01-701-1015 の Visit 8 (Week 8) の来院日 (Day 63) が、プロトコルで規定された許容期間 (Day 56±3日、すなわちDay 53-59) を超過していた。
        *   **プロトコル該当箇所:** Section 3.9 Visits (Page 9)
        *   **判断理由:** Visit 8 の実施日(SVSTDTC = 2014-03-05, Day 63) が、計画日(VISITDY = 56) に対する許容範囲 (±3日) を4日超過しているため。
        *   **判断根拠:**
            *   SV.USUBJID = 01-701-1015
            *   SV.VISITNUM = 8
            *   SV.VISIT = WEEK 8
            *   SV.VISITDY = 56
            *   SV.SVSTDTC = 2014-03-05
            *   DM.USUBJID = 01-701-1015
            *   DM.RFSTDTC = 2014-01-02 (計算上のStudy Day 1)
            *   Study Day of Visit = 63 (2014-03-05 - 2014-01-02 + 1)

    *   **逸脱No.:** 5
        *   **臨床試験結果への影響度合い:** Major
        *   **逸脱内容:** 被験者 01-701-1015 の Visit 10 (Week 16) の来院日 (Day 126) が、プロトコルで規定された許容期間 (Day 112±4日、すなわちDay 108-116) を大幅に超過していた。
        *   **プロトコル該当箇所:** Section 3.9 Visits (Page 9)
        *   **判断理由:** Visit 10 の実施日(SVSTDTC = 2014-05-07, Day 126) が、計画日(VISITDY = 112) に対する許容範囲 (±4日) を10日超過しているため。
        *   **判断根拠:**
            *   SV.USUBJID = 01-701-1015
            *   SV.VISITNUM = 10
            *   SV.VISIT = WEEK 16
            *   SV.VISITDY = 112
            *   SV.SVSTDTC = 2014-05-07
            *   DM.USUBJID = 01-701-1015
            *   DM.RFSTDTC = 2014-01-02 (計算上のStudy Day 1)
            *   Study Day of Visit = 126 (2014-05-07 - 2014-01-02 + 1)

3. 医療機関に問い合わせるクエリ: あり
    *   **クエリNo.:** 1
        *   **臨床試験結果への影響度合い:** Critical
        *   **医療機関への問い合わせ文面:** 被験者 01-701-1015 の同意取得日 (RFICDTC) が記録されていません。同意取得日を確認し、記録してください。同意取得が治験薬投与開始前に行われていることを確認してください。
        *   **判断理由:** 同意取得日の記録がなく、GCP遵守状況が確認できないため。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   DM.RFICDTC = ""
            *   プロトコル Section 3.4.2.1 Inclusion Criteria [6] (Page 12), Section 5.1 Informed Consent (Page 49)

    *   **クエリNo.:** 2
        *   **臨床試験結果への影響度合い:** Critical
        *   **医療機関への問い合わせ文面:** 被験者 01-701-1015 で報告された有害事象「DIARRHOEA」(AESEQ=3) は重篤 (Serious, AESHOSP=Y) と記録されています。SAEとして適切に報告されているか確認し、報告記録を提供してください。
        *   **判断理由:** 重篤な有害事象の適切な報告はGCPおよび規制要件上必須であるが、データ上では報告状況が確認できないため。
        *   **判断根拠:**
            *   AE.USUBJID = 01-701-1015
            *   AE.AESEQ = 3
            *   AE.AETERM = DIARRHOEA
            *   AE.AESER = Y
            *   AE.AESHOSP = Y
            *   プロトコル Section 3.9.3.2.2 Serious Adverse Events (Page 31)

    *   **クエリNo.:** 3
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 被験者 01-701-1015 の有害事象「DIARRHOEA」(AESEQ=3) について、開始日 (AESTDTC=2014-01-11) が終了日 (AEENDTC=2014-01-09) より後になっています。正しい開始日と終了日を確認し、データを修正してください。
        *   **判断理由:** 有害事象の開始日と終了日の関係が論理的に矛盾しており、データの信頼性に影響するため。
        *   **判断根拠:**
            *   AE.USUBJID = 01-701-1015
            *   AE.AESEQ = 3
            *   AE.AETERM = DIARRHOEA
            *   AE.AESTDTC = 2014-01-11
            *   AE.AEENDTC = 2014-01-09

    *   **クエリNo.:** 4
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 被験者 01-701-1015 は女性(SEX=F)ですが、選択基準[1]を満たすために閉経後(postmenopausal)である必要があります。閉経後であることを確認してください。
        *   **判断理由:** 選択基準の確認のため。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   DM.SEX = F
            *   プロトコル Section 3.4.2.1 Inclusion Criteria [1] (Page 11)

    *   **クエリNo.:** 5
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 選択基準[5]に基づき、被験者 01-701-1015 の過去1年以内のCNSイメージング（CTまたはMRI）が実施され、結果がADと矛盾しないことを確認してください。実施日と結果の概要を提供してください。
        *   **判断理由:** 選択基準の確認のため。データ上では確認できない。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   プロトコル Section 3.4.2.1 Inclusion Criteria [5] (Page 11-12)

    *   **クエリNo.:** 6
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 除外基準[16b]に基づき、被験者 01-701-1015 のスクリーニング時（Visit 1）のECG結果を確認してください。除外基準に記載された所見がないことを確認してください。
        *   **判断理由:** 除外基準の確認のため。ECGデータが提供されていない。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [16b] (Page 13)

    *   **クエリNo.:** 7
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** プロトコル逸脱No.3で指摘した通り、Ambulatory ECGの実施時期がプロトコル規定と異なります。逸脱の理由と、安全性評価への影響について説明してください。また、Ambulatory ECGの結果（特に洞停止、AVブロック、心室頻拍の有無）を確認し報告してください。
        *   **判断理由:** 安全性評価の重要な手順における逸脱であり、その理由と結果の確認が必要なため。
        *   **判断根拠:**
            *   SV.USUBJID = 01-701-1015
            *   SV.VISITNUM = 3.5, 6
            *   VS.USUBJID = 01-701-1015
            *   VS.VISITNUM = 3.5, 6
            *   プロトコル Section 3.1 (Page 8), Section 3.9.3.4.2 (Page 35), Section 3.9.4 (Page 36), Attachment LZZT.1 (Page 53)

    *   **クエリNo.:** 8
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** プロトコル逸脱No.5で指摘した通り、Visit 10の来院が計画日より10日遅延しています。遅延の理由と、治験薬投与状況、評価への影響について説明してください。
        *   **判断理由:** 評価スケジュールの許容範囲を大幅に超過しており、理由と影響の確認が必要なため。
        *   **判断根拠:**
            *   SV.USUBJID = 01-701-1015
            *   SV.VISITNUM = 10
            *   SV.SVSTDTC = 2014-05-07 (Day 126)
            *   SV.VISITDY = 112
            *   プロトコル Section 3.9 (Page 9)

    *   **クエリNo.:** 9
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** プロトコルスケジュール表(Attachment LZZT.1)に基づき、ECGはVisit 1, 4, 5, 7, 8, 9, 10, 11, 12, 13 で実施予定ですが、データがありません。ECGが予定通り実施されたか確認し、結果を提供してください。実施されていない場合は理由を説明してください。
        *   **判断理由:** 安全性評価項目であるECGの実施状況が不明なため。
        *   **判断根拠:**
            *   EGドメインデータなし
            *   プロトコル Attachment LZZT.1 (Page 53, 54), Section 3.9.3.4.2 (Page 35)

    *   **クエリNo.:** 10
        *   **臨床試験結果への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** プロトコルスケジュール表(Attachment LZZT.1)に基づき、各来院時に有害事象の確認が予定されていますが、AEドメインにはVisit 4相当日の記録しかありません。他のVisitでも有害事象の有無が確認されているか確認してください。確認されている場合、記録漏れであれば追記してください。
        *   **判断理由:** 安全性モニタリングの基本である有害事象の収集状況が不明確なため。
        *   **判断根拠:**
            *   AEデータ記録がVisit 4相当日のみ
            *   プロトコル Attachment LZZT.1 (Page 53, 54), Section 3.9.3.2 (Page 30)

    *   **クエリNo.:** 11
        *   **臨床試験結果への影響度合い:** Minor
        *   **医療機関への問い合わせ文面:** 除外基準[28b]に基づき、スクリーニング時（Visit 1）の Folate 検査結果を確認してください。検査が実施されていない場合は理由を説明してください。
        *   **判断理由:** 除外基準の確認のため。データが提供されていない。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   LBデータにFolate検査結果なし
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [28b] (Page 15)

    *   **クエリNo.:** 12
        *   **臨床試験結果への影響度合い:** Minor
        *   **医療機関への問い合わせ文面:** 除外基準[28b]に基づき、スクリーニング時（Visit 1）の甲状腺機能検査（TSH以外、例：Free thyroid index, T3 Uptake, T4）の結果を確認してください。検査が実施されていない場合は理由を説明してください。
        *   **判断理由:** 除外基準の確認のため。データが一部提供されていない。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   LBデータにTSH以外の甲状腺機能検査結果なし
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [28b] (Page 15-16)

    *   **クエリNo.:** 13
        *   **臨床試験結果への影響度合い:** Minor
        *   **医療機関への問い合わせ文面:** 除外基準[29b]に基づき、スクリーニング時（Visit 1）の梅毒検査の結果を確認してください。検査が実施されていない場合は理由を説明してください。
        *   **判断理由:** 除外基準の確認のため。データが提供されていない。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   LBデータに梅毒検査結果なし
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [29b] (Page 16)

    *   **クエリNo.:** 14
        *   **臨床試験結果への影響度合い:** Minor
        *   **医療機関への問い合わせ文面:** 除外基準[30b]に関連し、被験者 01-701-1015 が糖尿病（特にIDDM）であるか確認してください。該当する場合、またはスクリーニング時の血糖値が200mg/dLを超えていた場合、HbA1c検査の結果を確認してください。検査が実施されていない場合は理由を説明してください。
        *   **判断理由:** 除外基準の確認のため。データが提供されていない。
        *   **判断根拠:**
            *   DM.USUBJID = 01-701-1015
            *   MHデータに糖尿病の記録なし
            *   LBデータにHbA1c検査結果なし
            *   LB.LBTESTCD = GLUC, LB.VISITNUM = 1, LB.LBORRES = 85 (mg/dL) -> 200mg/dL以下
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [30b] (Page 16)

,Subject,Task1,Task2,Task3
0,01-701-1015,1. 症例サマリー：01-701-1015\n * 2013年12月26日 (Da...,1. 確認した症例：01-701-1015\n\n2. 医療機関に問い合わせるクエリ: あり...,1. 確認した症例：01-701-1015\n\n2. プロトコル逸脱: あり\n *...


## 結果の出力

### JSONの場合の出力関数

In [166]:
import re
import json

def extract_json(text):
    # ```json ... ``` のパターンを検索 (非貪欲マッチ)
    match = re.search(r'```json\s*([\s\S]*?)\s*```', text)

    if match:
        json_string = match.group(1)
        try:
            data = json.loads(json_string)
            return data
        except json.JSONDecodeError:
            print("Error: Invalid JSON found.")
            return None
    else:
        print("Error: No JSON code block found.")
        return None

# Test
#extract_json(df_results['Task1'][0])


In [167]:
import json
from datetime import datetime

# --- 新しいヘルパー関数 ---
def format_partial_date(date_str):
    """
    部分日付を含む日付文字列を可能な限り指定の日本語形式にフォーマットする。
    対応形式: YYYY-MM-DD, YYYY-MM, YYYY
    """
    if not date_str:
        return '日付不明'

    # 優先度順にフォーマットを試す
    formats_map = {
        '%Y-%m-%d': '%Y年%m月%d日',
        '%Y-%m': '%Y年%m月',
        '%Y': '%Y年'
    }

    for input_format, output_format in formats_map.items():
        try:
            date_obj = datetime.strptime(date_str, input_format)
            return date_obj.strftime(output_format)
        except ValueError:
            continue # 次のフォーマットを試す

    # どの形式にも一致しない場合は、元の文字列をそのまま返す
    # (予期しない形式や "UNKNOWN" などの文字列に対応するため)
    return date_str
# --- ヘルパー関数ここまで ---

def format_variables_list(variables):
    """
    変数リストをMarkdownの箇条書き形式の複数行文字列にフォーマットする関数
    各行は '* 変数名 = 値' の形式
    """
    if not variables:
        return [] # 空のリストを返す
    lines = [f"* {v.get('variable', 'N/A')} = {v.get('value', 'N/A')}" for v in variables]
    return lines

def indent_lines(lines, indent_spaces):
    """指定された行リストの各行にインデントを追加する"""
    indent = " " * indent_spaces
    return "\n".join([indent + line for line in lines])

def generate_queries_section(queries, section_title="医療機関に問い合わせるクエリ", no_label="クエリNo."):
    """
    クエリセクションのMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    markdown = []
    has_queries = bool(queries) # None や空リストでないかチェック

    markdown.append(f"{section_title}: {'あり' if has_queries else 'なし'}")
    if has_queries:
        for query in queries:
            markdown.append(f"    *   **{no_label}:** {query.get('query_no', 'N/A')}")
            markdown.append(f"        *   **臨床試験結果への影響度合い:** {query.get('criticality', 'N/A')}")
            markdown.append(f"        *   **医療機関への問い合わせ文面:** {query.get('inquiry', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {query.get('reason', 'N/A')}")
            markdown.append(f"        *   **判断根拠:**")
            variable_lines = format_variables_list(query.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))

    return "\n".join(markdown)

# --- generate_timeline_markdown を修正 ---
def generate_timeline_markdown(data):
    """
    'timeline' キーが存在する場合のMarkdownを生成する関数 (部分日付対応)
    """
    usubjid = data.get('usubjid', 'N/A')
    timeline = data.get('timeline', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 症例サマリー：{usubjid}")

    for entry in timeline:
        # 新しいヘルパー関数を使って日付をフォーマット
        formatted_date = format_partial_date(entry.get('date'))

        day = entry.get('day', '不明') # Day は日付形式に関わらず表示
        details = entry.get('details', '詳細不明')
        markdown.append(f"    *   {formatted_date} (Day {day}): {details}")

    markdown.append("") # 空行
    markdown.append(generate_queries_section(queries, section_title="2. 疑義事項", no_label="クエリNo."))

    return "\n".join(markdown)
# --- generate_timeline_markdown の修正ここまで ---

def generate_data_issues_markdown(data):
    """
    'data_issues' キーが存在する場合のMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    usubjid = data.get('usubjid', 'N/A')
    data_issues = data.get('data_issues', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 確認した症例：{usubjid}")
    markdown.append("") # 空行

    markdown.append(generate_queries_section(queries, section_title="2. 医療機関に問い合わせるクエリ", no_label="クエリNo."))
    markdown.append("") # 空行

    has_data_issues = bool(data_issues)
    markdown.append(f"3. 医療機関に問い合わせない疑義事項: {'あり' if has_data_issues else 'なし'}")
    if has_data_issues:
        for issue in data_issues:
            markdown.append(f"    *   **疑義No.:** {issue.get('issue_no', 'N/A')}")
            markdown.append(f"        *   **疑義事項:** {issue.get('inconsistency', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {issue.get('cause', 'N/A')}")
            markdown.append(f"        *   **判断根拠:")
            variable_lines = format_variables_list(issue.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))

    return "\n".join(markdown)

def generate_deviations_markdown(data):
    """
    'deviations' キーが存在する場合のMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    usubjid = data.get('usubjid', 'N/A')
    deviations = data.get('deviations', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 確認した症例：{usubjid}")
    markdown.append("") # 空行

    has_deviations = bool(deviations)
    markdown.append(f"2. プロトコル逸脱: {'あり' if has_deviations else 'なし'}")
    if has_deviations:
        for deviation in deviations:
            markdown.append(f"    *   **逸脱No.:** {deviation.get('deviation_no', 'N/A')}")
            markdown.append(f"        *   **臨床試験結果への影響度合い:** {deviation.get('impact', 'N/A')}")
            markdown.append(f"        *   **逸脱内容:** {deviation.get('description', 'N/A')}")
            markdown.append(f"        *   **プロトコル該当箇所:** {deviation.get('protocol_reference', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {deviation.get('justification', 'N/A')}")
            markdown.append(f"        *   **判断根拠:")
            variable_lines = format_variables_list(deviation.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))
    markdown.append("") # 空行

    markdown.append(generate_queries_section(queries, section_title="3. 医療機関に問い合わせるクエリ", no_label="クエリNo."))

    return "\n".join(markdown)

def json_to_markdown(json_input):
    """
    JSONデータを受け取り、指定の形式のMarkdownに変換するメイン関数 (部分日付対応)
    """
    try:
        if isinstance(json_input, str):
            data = json.loads(json_input)
        elif isinstance(json_input, dict):
            data = json_input
        else:
            return "エラー: 入力はJSON文字列またはPython辞書である必要があります。"
    except json.JSONDecodeError:
        return "エラー: 無効なJSON文字列です。"
    except Exception as e:
        return f"エラー: 予期せぬエラーが発生しました - {e}"

    if 'timeline' in data:
        return generate_timeline_markdown(data)
    elif 'data_issues' in data:
        return generate_data_issues_markdown(data)
    elif 'deviations' in data:
        return generate_deviations_markdown(data)
    else:
        usubjid = data.get('usubjid', 'N/A')
        queries = data.get('queries')
        markdown = []
        markdown.append(f"1. 確認した症例：{usubjid}")
        markdown.append("\n---\n")
        markdown.append("入力データには timeline, data_issues, deviations のいずれのキーも含まれていません。")
        if queries is not None:
             markdown.append("\n---\n")
             markdown.append(generate_queries_section(queries, section_title="クエリ情報", no_label="クエリNo."))
        return "\n".join(markdown)

# Test
#markdown_output = json_to_markdown(extract_json(df_results['Task3'][0]))
#print(markdown_output)


In [168]:
def result_output_JSON(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = json_to_markdown(extract_json(row['Task1']))
        task2 = json_to_markdown(extract_json(row['Task2']))
        task3 = json_to_markdown(extract_json(row['Task3']))

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data

#output_text = result_output_JSON(df_results)
#print(output_text)

### Markdownの場合の出力関数

In [169]:

def result_output_Markdown(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = row['Task1']
        task2 = row['Task2']
        task3 = row['Task3']

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data

#output_text = result_output_Markdown(df_results)

### 出力

In [170]:
if Output_Format == "Markdown":
  output_text = result_output_Markdown(df_results)
elif Output_Format == "JSON":
  output_text = result_output_JSON(df_results)

# mdファイルに保存
output_file = 'output_' + ModelName + '.md'  # 保存するファイル名を指定
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(output_text)

print(output_text)

# 01-701-1015
## Task1: Clinical Review Results
1. 症例サマリー：01-701-1015
    *   2013年12月26日 (Day -7): スクリーニング検査にて、アルカリホスファターゼ(ALP)低値(34 U/L, 基準値35-115)、アニソサイトーシス(Anisocytes)異常(Grade 1)、アスパラギン酸アミノトランスフェラーゼ(AST)高値(40 U/L, 基準値9-34)を認めた。身長147.32cm、体重53.98kg。脈拍は臥位57回/分、立位62-65回/分と徐脈傾向。血圧は臥位131/64 mmHg、立位129-147/57-83 mmHg。教育歴16年。アルツハイマー病と診断 (2010年4月30日発症)。既往歴に動悸、頭痛、耳鳴、胸やけ、脚のしびれ、子宮部分摘出術(1986年)、胆石(2012年)、甲状腺部分切除術(1973年)、咽頭痛(2013年12月)、扁桃摘出術(1973年)あり。併用薬としてアスピリン(2003年～)、カルシウム(2013年～)、プレマリン(2006年～)、タイレノール(2003年～)を使用中。MMSE 23点 (Day -7, Visit 1)。
    *   2013年12月31日 (Day -2): スクリーニング検査。脈拍は臥位56回/分、立位57-60回/分。血圧は臥位138/68 mmHg、立位137-145/59-71 mmHg。
    *   2014年01月02日 (Day 1): ベースライン。プラセボ投与開始。体重54.43kg。脈拍は臥位56回/分、立位59回/分。血圧は臥位130/56 mmHg、立位121-131/51-61 mmHg。ADAS-Cog(11)合計点13点。NPI-X合計点0点。DADスコア全項目「Yes」。
    *   2014年01月03日 (Day 2): 投与部位紅斑(軽度)、投与部位掻痒(軽度)発現。治験薬との関連あり。併用薬としてネオスポリン（外用）開始。
    *   2014年01月11日 (Day 10): 下痢(軽度)発現。本有害事象は重篤（入院）と判断された。
    *   2014年01月16日 (Day 15): Week 2 Visit。アラニンアミノトランスフェラーゼ(

# 以下メモ

In [171]:
# Schema memo
'''
{
  "name": "Clinical_Review",
  "description": "Schema for clinical case summaries and associated queries",
  "strict": true,
  "schema": {
    "type": "object",
    "properties": {
      "usubjid": {
        "type": "string",
        "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)"
      },
      "timeline": {
        "type": ["array", "null"],
        "description": "Chronological events in the subject's case",
        "items": {
          "type": "object",
          "properties": {
            "date": {
              "type": "string",
              "description": "Date of the event.  Allows full (YYYY-MM-DD), partial (YYYY-MM), or year-only (YYYY) formats."
            },
            "day": {
              "type": "integer",
              "description": "Day relative to study start (Day 1).  Allows negative values for pre-treatment days."
            },
            "details": {
              "type": "string",
              "description": "Detailed information about the event (e.g., adverse event, medication administration)"
            }
          },
          "required": [
            "date",
            "day",
            "details"
          ],
          "additionalProperties": false
        }
      },
      "data_issues": {
        "type": ["array", "null"],
        "description": "List of data issues identified during review",
        "items": {
          "type": "object",
          "properties": {
            "issue_no": {
              "type": "integer",
              "description": "Unique issue number"
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the issue",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            },
            "inconsistency": {
              "type": "string",
              "description": "Description of the data inconsistency"
            },
            "cause": {
              "type": "string",
              "description": "Suspected cause of the issue"
            },
            "resolution": {
              "type": "string",
              "description": "Proposed resolution for the issue"
            }
          },
          "required": [
            "issue_no",
            "variables",
            "inconsistency",
            "cause",
            "resolution"
          ],
          "additionalProperties": false
        }
      },
      "deviations": {
        "type": ["array", "null"],
        "description": "List of protocol deviations",
        "items": {
          "type": "object",
          "properties": {
            "deviation_no": {
              "type": "integer",
              "description": "Unique deviation number"
            },
            "impact": {
              "type": "string",
              "description": "Impact of the deviation on the clinical trial results",
              "enum": ["Critical", "Major", "Minor"]
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the deviation",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            },
            "description": {
              "type": "string",
              "description": "Description of the deviation"
            },
            "protocol_reference": {
              "type": "string",
              "description": "Reference to the relevant section in the protocol"
            },
            "justification": {
              "type": "string",
              "description": "Justification for the deviation classification"
            }
          },
          "required": [
            "deviation_no",
            "impact",
            "variables",
            "description",
            "protocol_reference",
            "justification"
          ],
          "additionalProperties": false
        }
      },
      "queries": {
        "type": ["array", "null"],
        "description": "List of queries related to the subject",
        "items": {
          "type": "object",
          "properties": {
            "query_no": {
              "type": "integer",
              "description": "Unique query number"
            },
            "criticality": {
              "type": "string",
              "description": "Criticality of the query on the clinical trial results",
              "enum": ["Critical", "Major", "Minor"]
            },
            "inquiry": {
              "type": "string",
              "description": "Text of the inquiry to the study site"
            },
            "reason": {
              "type": "string",
              "description": "Justification for the inquiry"
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the query",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            }
          },
          "required": [
            "query_no",
            "criticality",
            "inquiry",
            "reason",
            "variables"
          ],
          "additionalProperties": false
        }
      }
    },
    "required": [
      "usubjid",
      "timeline",
      "data_issues",
      "deviations",
      "queries"
    ],
    "additionalProperties": false
  }
}

''

SyntaxError: incomplete input (<ipython-input-171-2de6c870828f>, line 2)